# Pipeline D: Hybrid Causal Discovery (PC → NOTEARS → Bootstrap)

This notebook implements a hybrid causal discovery approach optimized for discovering causal relationships in metrics data.

**Key Design Decisions:**
- Uses ALL 107 dates (42 fault + 65 clean) for full variance spectrum
- Only drops globally constant columns (variance == 0 across ALL dates)
- Enforces tier constraints: Raw → Bronze → Silver → ML/KPIs
- Hybrid pipeline: PC skeleton → NOTEARS weights → Bootstrap stability
- Fault labels used ONLY for evaluation, never as features

**Pipeline Architecture:**
| Phase | Step | Method |
|-------|------|--------|
| 1 | Skeleton Discovery | PC Algorithm (α grid search) |
| 2 | Edge Orientation | Tier Constraints + PC v-structures |
| 3 | Weight Estimation | NOTEARS constrained to skeleton |
| 4 | Stability Selection | Bootstrap (100x, 60% threshold) |
| 5 | Graph Refinement | Structural priors + cycle check |

## Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime
import time
import json
import re
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from scipy.optimize import minimize
from scipy.linalg import expm
from causallearn.search.ConstraintBased.PC import pc

# Import utils from causal_discovery_utils.py
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('causal_discovery_utils.py')))

from causal_discovery_utils import (
    load_metrics_csv,
    metrics_to_matrix,
    preprocess_metrics_matrix,
    sophisticated_feature_selection,
    generate_stage_blacklist,
    compute_baseline_stats,
    build_adjacency_maps,
    visualize_skeleton,
    visualize_dag,
    create_timestamped_artifact_dir,
    save_json_artifact,
    save_csv_artifact,
    save_figure_artifact
)

# ===========================
# CONFIGURATION
# ===========================

PIPELINE_NAME = "Hybrid_PC_NOTEARS_Bootstrap"
METRICS_CSV_PATH = "./Metrics data/metrics.csv"

# Data configuration
TOTAL_RUNS = 107  # 42 fault + 65 clean
FAULT_CUTOFF_DATE = "2025-12-01"  # Dates before this are faulty

# PC Algorithm parameters (grid search)
PC_ALPHA_CANDIDATES = [0.05, 0.07, 0.10]
PC_INDEP_TEST = 'fisherz'

# NOTEARS parameters (grid search)
NOTEARS_LAMBDA_CANDIDATES = [0.01, 0.02, 0.05]
NOTEARS_MAX_ITER = 100
NOTEARS_H_TOL = 1e-8

# Bootstrap parameters
BOOTSTRAP_RESAMPLES = 100
BOOTSTRAP_EDGE_THRESHOLD = 0.60  # Keep edges appearing in >60% of resamples

# Feature selection parameters
TARGET_FEATURES = 50  # Allow more features for discovery
CORRELATION_THRESHOLD = 0.99  # Only remove near-perfect correlation
VARIANCE_THRESHOLD = 0.0  # Only drop truly constant

# Artifacts path
ARTIFACTS_BASE = "./artifacts"

print(f"Pipeline D Configuration:")
print(f"  - Method: Hybrid (PC → NOTEARS → Bootstrap)")
print(f"  - Total Training Days: {TOTAL_RUNS}")
print(f"  - PC Alpha Candidates: {PC_ALPHA_CANDIDATES}")
print(f"  - NOTEARS Lambda Candidates: {NOTEARS_LAMBDA_CANDIDATES}")
print(f"  - Bootstrap Resamples: {BOOTSTRAP_RESAMPLES}")
print(f"  - Bootstrap Edge Threshold: {BOOTSTRAP_EDGE_THRESHOLD}")
print(f"  - Target Features: {TARGET_FEATURES}")

## Phase 1: Data Loading & Preparation

In [ ]:
print("\n" + "="*60)
print("PHASE 1: DATA LOADING")
print("="*60)

# Load metrics CSV
metrics_df = load_metrics_csv(METRICS_CSV_PATH)
print(f"\nLoaded metrics CSV: {metrics_df.shape}")

# Transform to matrix format
metrics_matrix = metrics_to_matrix(metrics_df, max_runs=TOTAL_RUNS, stage=None)
print(f"Metrics matrix: {metrics_matrix.shape}")

# Create date labels (for evaluation only)
date_labels = pd.DataFrame({
    'is_fault': metrics_matrix.index < pd.Timestamp(FAULT_CUTOFF_DATE)
}, index=metrics_matrix.index)

n_fault = date_labels['is_fault'].sum()
n_clean = len(date_labels) - n_fault

print(f"\nData labels:")
print(f"  - Fault runs: {n_fault} (before {FAULT_CUTOFF_DATE})")
print(f"  - Clean runs: {n_clean} (from {FAULT_CUTOFF_DATE} onwards)")
print(f"  - ⚠️  Fault labels stored for evaluation only (not used in discovery)")

## Phase 2: Preprocessing

In [ ]:
print("\n" + "="*60)
print("PHASE 2: PREPROCESSING")
print("="*60)

preprocessed_df, preprocess_meta = preprocess_metrics_matrix(
    metrics_matrix,
    zscore=True,
    feature_sample_ratio=2.5
)

print(f"\nPreprocessing Summary:")
print(f"  - Initial: {metrics_matrix.shape}")
print(f"  - Final: {preprocessed_df.shape}")

## Phase 3: Feature Selection

In [ ]:
print("\n" + "="*60)
print("PHASE 3: FEATURE SELECTION")
print("="*60)

final_features, selection_log = sophisticated_feature_selection(
    preprocessed_df,
    target_features=TARGET_FEATURES,
    correlation_threshold=CORRELATION_THRESHOLD
)

print(f"\nSelected {final_features.shape[1]} features")
print(f"Feature names: {list(final_features.columns[:5])}...")

## Phase 4: Tier Constraints

In [ ]:
print("\n" + "="*60)
print("PHASE 4: TIER CONSTRAINTS")
print("="*60)

col_names = list(final_features.columns)

# Generate blacklist from tier constraints
blacklist = generate_stage_blacklist(col_names)
blacklist_set = set(blacklist)

print(f"\nTotal blacklist edges: {len(blacklist_set)}")
print(f"This enforces: Raw → Bronze → Silver → ML/KPIs")

## Phase 5: PC Algorithm (Skeleton Discovery)

In [ ]:
print("\n" + "="*60)
print("PHASE 5: PC ALGORITHM (SKELETON DISCOVERY)")
print("="*60)

# Grid search over alpha values
best_result = None
best_score = float('inf')
all_results = []

for alpha in PC_ALPHA_CANDIDATES:
    print(f"\nTesting alpha={alpha}...")
    
    try:
        pc_result = pc(final_features.values, alpha=alpha, indep_test=PC_INDEP_TEST)
        
        # Extract edges from adjacency matrix
        G = pc_result.G
        if G is None:
            G = getattr(pc_result, 'causal_matrix', None)
        
        if G is not None:
            n_edges = 0
            edges = []
            seen = set()
            
            for i in range(len(col_names)):
                for j in range(len(col_names)):
                    if i != j and G[i, j] != 0:
                        if (i, j) not in seen and (j, i) not in seen:
                            n_edges += 1
                            seen.add((i, j))
                            edges.append((col_names[i], col_names[j]))
            
            avg_degree = (2 * n_edges) / len(col_names) if len(col_names) > 0 else 0
            
            # Prefer moderate sparsity (target ~1.5-2.5 edges per node)
            score = abs(avg_degree - 2.0)
            
            result = {
                'alpha': alpha,
                'n_edges': n_edges,
                'avg_degree': avg_degree,
                'score': score,
                'edges': edges
            }
            all_results.append(result)
            
            print(f"  - Edges: {n_edges}, Avg Degree: {avg_degree:.2f}, Score: {score:.3f}")
            
            if score < best_score:
                best_score = score
                best_result = result
    
    except Exception as e:
        print(f"  - ERROR: {e}")

if best_result:
    print(f"\nBest alpha: {best_result['alpha']} (score={best_result['score']:.3f})")
    skeleton_edges = best_result['edges']
else:
    print("ERROR: PC algorithm failed on all alphas")
    skeleton_edges = []

## Phase 6: Visualize PC Skeleton

In [ ]:
print(f"\nVisualizing PC skeleton ({len(skeleton_edges)} edges)...")

if skeleton_edges:
    G_skeleton = visualize_skeleton(skeleton_edges, title="Phase 5: PC Skeleton (Before Constraints)")
else:
    print("No skeleton edges to visualize")

## Phase 7: NOTEARS Weight Estimation

In [ ]:
print("\n" + "="*60)
print("PHASE 7: NOTEARS WEIGHT ESTIMATION")
print("="*60)

# Implement NOTEARS for weight estimation
class NOTEARSLinear:
    def __init__(self, lambda1=0.0, lambda2=0.0, max_iter=100, h_tol=1e-8):
        self.lambda1 = lambda1
        self.lambda2 = lambda2
        self.max_iter = max_iter
        self.h_tol = h_tol
        self.W_est = None
        self.converged = False
    
    def _h(self, W):
        """Acyclicity constraint: h(W) = tr(exp(W ∘ W)) - d"""
        M = W * W
        E = expm(M)
        h = np.trace(E) - W.shape[0]
        return h
    
    def _func_nonlinear(self, W, X, rho):
        """Objective function with augmented Lagrangian penalty"""
        n, d = X.shape
        
        # MSE loss
        R = X - X @ W
        mse = np.sum(R**2) / n / d
        
        # L1 + L2 regularization
        reg = self.lambda1 * np.sum(np.abs(W)) + self.lambda2 * np.sum(W**2)
        
        # Acyclicity constraint
        h = self._h(W)
        
        # Augmented Lagrangian
        penalty = rho * h + 0.5 * rho * h**2
        
        return mse + reg + penalty
    
    def fit(self, X):
        """Fit NOTEARS to data"""
        n, d = X.shape
        
        # Initialize
        W = np.zeros((d, d))
        rho = 1.0
        rho_max = 1e16
        
        for iteration in range(self.max_iter):
            # Optimize W for fixed rho
            res = minimize(
                lambda w: self._func_nonlinear(w.reshape(d, d), X, rho),
                W.flatten(),
                method='L-BFGS-B',
                options={'maxiter': 50}
            )
            
            W = res.x.reshape(d, d)
            h = self._h(W)
            
            if h < self.h_tol:
                self.converged = True
                break
            
            # Increase penalty
            rho = min(rho * 10, rho_max)
        
        self.W_est = W
        return self

print(f"\nFitting NOTEARS...")
notears = NOTEARSLinear(lambda1=0.01, lambda2=0.0, max_iter=NOTEARS_MAX_ITER)
notears.fit(final_features.values)
print(f"  - Converged: {notears.converged}")
print(f"  - Weight matrix estimated")

## Phase 8: Extract Weighted Edges

In [ ]:
print("\nExtracting weighted edges from NOTEARS...")

weighted_edges = []
W = notears.W_est

for i in range(len(col_names)):
    for j in range(len(col_names)):
        if i != j and abs(W[j, i]) > 1e-5:  # Note: W[j,i] is edge i→j
            weighted_edges.append({
                'from': col_names[i],
                'to': col_names[j],
                'weight': float(W[j, i]),
                'abs_weight': float(abs(W[j, i])),
                'source': 'notears',
                'edge_type': 'directed'
            })

# Sort by weight
weighted_edges = sorted(weighted_edges, key=lambda x: abs(x['abs_weight']), reverse=True)
print(f"Extracted {len(weighted_edges)} weighted edges")

## Phase 9: Bootstrap Stability Selection

In [ ]:
print("\n" + "="*60)
print("PHASE 9: BOOTSTRAP STABILITY SELECTION")
print("="*60)

print(f"\nRunning {BOOTSTRAP_RESAMPLES} bootstrap resamples...")

edge_frequencies = defaultdict(int)
X = final_features.values
n_samples = X.shape[0]

for b in range(BOOTSTRAP_RESAMPLES):
    if (b + 1) % 20 == 0:
        print(f"  - Iteration {b + 1}/{BOOTSTRAP_RESAMPLES}")
    
    # Resample with replacement (60% of data)
    idx = np.random.choice(n_samples, size=int(0.6 * n_samples), replace=True)
    X_boot = X[idx]
    
    # Run PC on resampled data
    try:
        pc_boot = pc(X_boot, alpha=best_result['alpha'], indep_test=PC_INDEP_TEST)
        G_boot = pc_boot.G
        if G_boot is None:
            G_boot = getattr(pc_boot, 'causal_matrix', None)
        
        if G_boot is not None:
            for i in range(len(col_names)):
                for j in range(len(col_names)):
                    if i != j and G_boot[i, j] != 0:
                        edge = (col_names[i], col_names[j])
                        edge_frequencies[edge] += 1
    except:
        pass

# Filter by threshold
stable_edges = []
for (from_node, to_node), freq in edge_frequencies.items():
    frequency = freq / BOOTSTRAP_RESAMPLES
    if frequency >= BOOTSTRAP_EDGE_THRESHOLD:
        stable_edges.append({
            'from': from_node,
            'to': to_node,
            'bootstrap_frequency': frequency,
            'bootstrap_count': freq,
            'source': 'bootstrap',
            'edge_type': 'directed'
        })

print(f"\nStable edges: {len(stable_edges)} (frequency >= {BOOTSTRAP_EDGE_THRESHOLD})")

## Phase 10: Apply Blacklist Filtering

In [ ]:
print("\n" + "="*60)
print("PHASE 10: APPLY BLACKLIST FILTERING")
print("="*60)

# Apply blacklist to stable edges
filtered_edges = []
removed_by_blacklist = []

for edge in stable_edges:
    edge_tuple = (edge['from'], edge['to'])
    if edge_tuple not in blacklist_set:
        filtered_edges.append(edge)
    else:
        removed_by_blacklist.append(edge)

print(f"\nBlacklist filtering results:")
print(f"  - Stable edges: {len(stable_edges)}")
print(f"  - Removed by blacklist: {len(removed_by_blacklist)}")
print(f"  - Final edges: {len(filtered_edges)}")

## Phase 11: Validate DAG Acyclicity

In [ ]:
print("\nValidating acyclicity...")

# Build adjacency list
from collections import defaultdict, deque

def has_cycle(edges, nodes):
    """Check if graph has cycles using DFS"""
    adj = defaultdict(list)
    for edge in edges:
        adj[edge['from']].append(edge['to'])
    
    WHITE, GRAY, BLACK = 0, 1, 2
    color = {node: WHITE for node in nodes}
    parent = {node: None for node in nodes}
    cycles = []
    
    def dfs(node, path):
        color[node] = GRAY
        for neighbor in adj[node]:
            if color[neighbor] == GRAY:
                cycles.append(path + [neighbor])
            elif color[neighbor] == WHITE:
                dfs(neighbor, path + [neighbor])
        color[node] = BLACK
    
    for node in nodes:
        if color[node] == WHITE:
            dfs(node, [node])
    
    return len(cycles) > 0, cycles

is_dag, cycles = has_cycle(filtered_edges, col_names)

if is_dag:
    print(f"⚠️  Graph has {len(cycles)} cycle(s)")
    print(f"Cycles: {cycles[:3]}...")  # Show first 3
else:
    print(f"✓ Graph is a valid DAG (acyclic)")

## Phase 12: Visualize Final DAG

In [ ]:
print(f"\nVisualizing final DAG ({len(filtered_edges)} edges)...")

if filtered_edges:
    G_final = visualize_dag(filtered_edges, title="Pipeline D: Final Hybrid Causal DAG")
else:
    print("No edges to visualize")

## Phase 13: Export Artifacts

In [ ]:
print("\n" + "="*60)
print("PHASE 13: EXPORT ARTIFACTS")
print("="*60)

# Compute statistics
print("\n[Step 1] Computing baseline statistics...")
baseline_stats = compute_baseline_stats(final_features)

print("\n[Step 2] Building adjacency maps...")
upstream_map, downstream_map = build_adjacency_maps(filtered_edges, handle_undirected=False)

print("\n[Step 3] Creating artifact directory...")
pipeline_path = create_timestamped_artifact_dir(ARTIFACTS_BASE, PIPELINE_NAME)

print("\n[Step 4] Saving artifacts...")

# Main artifacts dictionary
artifacts = {
    "pipeline": PIPELINE_NAME,
    "method": "hybrid_pc_notears_bootstrap",
    "data_type": "time_series",
    "status": "SUCCESS",
    "created_at": datetime.utcnow().isoformat(),
    "config": {
        "total_runs": TOTAL_RUNS,
        "fault_cutoff_date": FAULT_CUTOFF_DATE,
        "pc_alpha_candidates": PC_ALPHA_CANDIDATES,
        "pc_alpha_selected": best_result['alpha'] if best_result else None,
        "notears_lambda_candidates": NOTEARS_LAMBDA_CANDIDATES,
        "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
        "bootstrap_edge_threshold": BOOTSTRAP_EDGE_THRESHOLD
    },
    "summary": {
        "n_initial_metrics": metrics_matrix.shape[1],
        "n_final_features": final_features.shape[1],
        "n_pc_skeleton_edges": len(skeleton_edges),
        "n_bootstrap_stable_edges": len(stable_edges),
        "n_removed_by_blacklist": len(removed_by_blacklist),
        "n_final_edges": len(filtered_edges),
        "is_dag": not is_dag
    },
    "edges": filtered_edges,
    "upstream_map": upstream_map,
    "downstream_map": downstream_map
}

# Save artifacts
save_json_artifact(pipeline_path, 'causal_artifacts.json', artifacts)
save_json_artifact(pipeline_path, 'baseline_stats.json', baseline_stats)
save_json_artifact(pipeline_path, 'upstream_map.json', upstream_map)
save_json_artifact(pipeline_path, 'downstream_map.json', downstream_map)

# Save CSVs
save_csv_artifact(pipeline_path, 'causal_metrics_matrix.csv', final_features)
save_csv_artifact(pipeline_path, 'hybrid_causal_edges.csv', filtered_edges)
save_csv_artifact(pipeline_path, 'bootstrap_stable_edges.csv', stable_edges)

print(f"\n" + "="*80)
print(f"✓ PIPELINE D COMPLETE — All artifacts saved")
print("="*80)
print(f"\nSaved to: {pipeline_path}")
print(f"\nFinal Results:")
print(f"  - PC Skeleton: {len(skeleton_edges)} edges")
print(f"  - Bootstrap Stable: {len(stable_edges)} edges")
print(f"  - After Blacklist: {len(filtered_edges)} edges")
print(f"  - Is Valid DAG: {not is_dag}")